## SGC Implementation
### SGC Model Class

The SGC model is a single linear layer (`nn.Linear`) applied to the precomputed features S²X. After graph propagation is done in the preprocessing step, the model itself is simply logistic regression one matrix multiply followed by a softmax.( PyTorch's CrossEntropyLoss applies softmax internally, so we did not need to explicitly call it in the forward pass. CrossEntropyLoss is calculated in benchmark.ipynb) This reflects the paper's core argument that the graph structure can be fully separated from the learning step.

### SGC Precompute Step

The precompute function builds the normalized adjacency matrix S = D̃⁻¹/²ÃD̃⁻¹/² by adding self-loops, computing degree normalization, and constructing a sparse adjacency tensor. It then applies sparse matrix multiplication (`torch.spmm`) K=2 times to propagate node features across the graph. This step runs once before training and its output is fixed for all epochs.

#### Installing Libraries

In [ ]:
# Installing libraries
%%capture
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torch_geometric
!pip install -q scipy


### SGC Model Class

In [ ]:
class SGC(nn.Module):
    """Single linear layer applied to precomputed S^K X."""
    def __init__(self, nfeat, nclass):
        super().__init__()
        self.lin = nn.Linear(nfeat, nclass)
    def forward(self, x):
        return self.lin(x)

### SGC Precompute Step

In [ ]:
def sgc_precompute(data, K=2):
    """Compute S^K X. Returns (features, precompute_time_seconds)."""
    t0 = time.perf_counter()
    edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)
    row, col = edge_index
    deg = degree(col, data.num_nodes, dtype=data.x.dtype)
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
    edge_weight = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    adj = torch.sparse_coo_tensor(edge_index, edge_weight,
                                   (data.num_nodes, data.num_nodes)).to(data.x.device)
    x = data.x
    for _ in range(K):
        x = torch.spmm(adj, x)
    if x.is_cuda: torch.cuda.synchronize()
    return x, time.perf_counter() - t0
